# Repository: Swappable Data Sources

This notebook covers:

1. Define an `AssetRepo` **Protocol** that describes how routes talk to storage
2. Implement two concrete repos: in-memory and SQLite
3. Wire the repo into routes via `Depends` so the handler doesn't know which one it got
4. Swap repos in tests with `app.dependency_overrides` — no monkeypatching, no global state

**Scope**: FastAPI + `sqlite3` (stdlib) + `TestClient`.

This is the pattern that makes a FastAPI app **portable across environments and trivially testable**. Production runs against Postgres. Local dev runs against SQLite. Unit tests run against an in-memory dict. The handler code never changes.

## 1. Why a Repository Protocol

A handler that does `conn.execute("INSERT INTO assets ...")` is welded to one database. Three problems pile up fast:

- **Tests are slow or flaky.** They need a real database. Setup/teardown isn't free.
- **Local dev is heavy.** Every contributor has to run Postgres locally.
- **Refactoring is expensive.** Want to add caching? Change the schema? You're touching every handler.

The fix is to put a thin abstraction between the handler and the database:

```
handler ──▶ AssetRepo (interface) ◀── InMemoryAssetRepo
                                  ◀── SQLiteAssetRepo
                                  ◀── PostgresAssetRepo
```

The handler depends on the **shape** (`AssetRepo`), not the implementation. Implementations are swapped via `Depends`. In Python the "interface" part is a `typing.Protocol` — duck-typing with a name, no inheritance required.

## 2. Defining the Protocol

`Protocol` documents the methods a `Repo` must provide. Any class that has the same methods with compatible signatures is a valid `AssetRepo` — no `class XRepo(AssetRepo)` boilerplate. Static type checkers will flag missing methods; at runtime there's zero overhead.

In [ ]:
from typing import Protocol, Iterable
from pydantic import BaseModel, Field

class Asset(BaseModel):
    id: int | None = None  # assigned by the repo on create
    ticker: str = Field(min_length=1, max_length=10, pattern=r"^[A-Z.]+$")
    name: str = Field(min_length=1)
    price: float = Field(ge=0)

class AssetRepo(Protocol):
    def list(self) -> list[Asset]: ...
    def get(self, asset_id: int) -> Asset | None: ...
    def add(self, asset: Asset) -> Asset: ...
    def delete(self, asset_id: int) -> bool: ...

# A Protocol is a static-typing concept — there's no class to instantiate.
# Concrete repos just need methods with these names and signatures.
print("Protocol declared. AssetRepo methods:", [m for m in dir(AssetRepo) if not m.startswith('_')])

## 3. In-Memory Implementation

Simplest possible backend: a dict keyed by id. Fast, deterministic, ideal for unit tests and learning. Production code never sees this implementation directly — but every test does.

In [ ]:
class InMemoryAssetRepo:
    def __init__(self):
        self._data: dict[int, Asset] = {}
        self._next_id = 1

    def list(self) -> list[Asset]:
        return list(self._data.values())

    def get(self, asset_id: int) -> Asset | None:
        return self._data.get(asset_id)

    def add(self, asset: Asset) -> Asset:
        stored = asset.model_copy(update={"id": self._next_id})
        self._data[self._next_id] = stored
        self._next_id += 1
        return stored

    def delete(self, asset_id: int) -> bool:
        return self._data.pop(asset_id, None) is not None

# Sanity check that it satisfies the Protocol — by behavior, not inheritance.
repo: AssetRepo = InMemoryAssetRepo()
a = repo.add(Asset(ticker="AAPL", name="Apple", price=189.5))
b = repo.add(Asset(ticker="MSFT", name="Microsoft", price=410.0))
print("listed:", [a.ticker for a in repo.list()])
print("by id 1:", repo.get(1))
print("delete id 2:", repo.delete(2), "remaining:", [a.ticker for a in repo.list()])

## 4. SQLite Implementation

Same Protocol, persistent storage. `sqlite3` ships with Python — no setup, no external service. Good enough for examples; production code would substitute Postgres / MySQL / DynamoDB with the same `Repo` shape.

We open one connection per repo instance and reuse it. In a web app you'd inject one repo per request (next section), which translates to one connection per request — the natural unit of work.

In [ ]:
import sqlite3

class SQLiteAssetRepo:
    def __init__(self, db_path: str = ":memory:"):
        # check_same_thread=False because FastAPI's threadpool may dispatch the
        # request that holds this repo on a different thread than the one that opened it.
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self.conn.row_factory = sqlite3.Row
        self.conn.execute('''
            CREATE TABLE IF NOT EXISTS assets (
                id     INTEGER PRIMARY KEY AUTOINCREMENT,
                ticker TEXT NOT NULL UNIQUE,
                name   TEXT NOT NULL,
                price  REAL NOT NULL
            )
        ''')

    def list(self) -> list[Asset]:
        rows = self.conn.execute("SELECT id, ticker, name, price FROM assets ORDER BY id").fetchall()
        return [Asset(**dict(r)) for r in rows]

    def get(self, asset_id: int) -> Asset | None:
        r = self.conn.execute("SELECT id, ticker, name, price FROM assets WHERE id = ?", (asset_id,)).fetchone()
        return Asset(**dict(r)) if r else None

    def add(self, asset: Asset) -> Asset:
        cur = self.conn.execute(
            "INSERT INTO assets (ticker, name, price) VALUES (?, ?, ?)",
            (asset.ticker, asset.name, asset.price),
        )
        self.conn.commit()
        return asset.model_copy(update={"id": cur.lastrowid})

    def delete(self, asset_id: int) -> bool:
        cur = self.conn.execute("DELETE FROM assets WHERE id = ?", (asset_id,))
        self.conn.commit()
        return cur.rowcount > 0

# Quick exercise: confirm the same Protocol works.
sql_repo: AssetRepo = SQLiteAssetRepo()  # :memory: db
sql_repo.add(Asset(ticker="GOOGL", name="Alphabet", price=140.0))
sql_repo.add(Asset(ticker="NVDA", name="NVIDIA", price=920.0))
print("listed:", [(a.id, a.ticker) for a in sql_repo.list()])

Notice what didn't change between the in-memory and SQLite versions:

- The Protocol — `list / get / add / delete` signatures are identical.
- The handler code we're about to write — it'll consume any `AssetRepo` without caring which.
- The `Asset` model — Pydantic doesn't know or care where the data came from.

This is the whole point: storage decisions stay inside the repo class. The rest of the app just asks for "an `AssetRepo`".

## 5. Injecting via `Depends`

The handler declares `repo: AssetRepo = Depends(get_repo)`. The provider `get_repo` decides which concrete repo to return. Most apps wire production to SQLite/Postgres and override in tests, but you can be cleverer — feature flags can pick between repos, per-tenant routing can return different repos, etc.

We'll go with the simplest: a module-level singleton repo, returned by `get_repo`. Per-request connection management would use a `yield` dep (the pattern from notebook 4.1).

In [ ]:
from fastapi import FastAPI, Depends, HTTPException
from fastapi.testclient import TestClient

app = FastAPI()

# Production wiring: one repo for the whole app.
_repo_singleton = SQLiteAssetRepo()  # :memory: for this notebook

def get_repo() -> AssetRepo:
    return _repo_singleton

@app.get("/assets", response_model=list[Asset])
def list_assets(repo: AssetRepo = Depends(get_repo)):
    return repo.list()

@app.get("/assets/{asset_id}", response_model=Asset)
def get_asset(asset_id: int, repo: AssetRepo = Depends(get_repo)):
    asset = repo.get(asset_id)
    if asset is None:
        raise HTTPException(status_code=404, detail="asset not found")
    return asset

@app.post("/assets", response_model=Asset, status_code=201)
def create_asset(asset: Asset, repo: AssetRepo = Depends(get_repo)):
    return repo.add(asset)

@app.delete("/assets/{asset_id}", status_code=204)
def delete_asset(asset_id: int, repo: AssetRepo = Depends(get_repo)):
    if not repo.delete(asset_id):
        raise HTTPException(status_code=404, detail="asset not found")

client = TestClient(app)

# Exercise the API end-to-end against the singleton SQLite repo.
print("POST AAPL:", client.post("/assets", json={"ticker": "AAPL", "name": "Apple", "price": 189.5}).json())
print("POST MSFT:", client.post("/assets", json={"ticker": "MSFT", "name": "Microsoft", "price": 410.0}).json())
print("GET list:", [a["ticker"] for a in client.get("/assets").json()])
print("GET 404:", client.get("/assets/9999").status_code)

The handlers above are completely **storage-agnostic** — they could be running against any class that satisfies `AssetRepo`. The Protocol is the seam, and `Depends(get_repo)` is the joint. Anywhere a route does `asset_db.execute("...")`, you've lost the ability to swap implementations cheaply.

## 6. Overriding the Repo in Tests

`app.dependency_overrides` is a dict on the app object. The key is the dep callable; the value is the replacement callable. While the override is in place, FastAPI uses the replacement everywhere `Depends(original)` appears.

This is the override mechanism we'll lean on heavily in chapter 7. It has three properties that matter:

- **No monkeypatching, no globals.** The override is scoped to the test and clears with `.clear()` (or via a fixture).
- **Works across the whole graph.** If `get_repo` is itself referenced by `get_authenticated_repo`, that chain still picks up the override.
- **The handler is unaware.** No `if testing:` branches anywhere.

In [ ]:
# Swap to an in-memory repo for "tests".
test_repo = InMemoryAssetRepo()
app.dependency_overrides[get_repo] = lambda: test_repo

test_client = TestClient(app)

print("starts empty:", test_client.get("/assets").json())
test_client.post("/assets", json={"ticker": "TSLA", "name": "Tesla", "price": 250.0})
test_client.post("/assets", json={"ticker": "META", "name": "Meta", "price": 480.0})
print("with two:", [(a["id"], a["ticker"]) for a in test_client.get("/assets").json()])

# Clear the override. The original singleton SQLite repo is in charge again.
app.dependency_overrides.clear()
print("after clear, back to SQLite:", [a["ticker"] for a in client.get("/assets").json()])

Three things to lock in:

- The in-memory repo started empty and accepted writes — completely isolated from the SQLite singleton.
- After `clear()`, the next request went back to the SQLite repo and its existing data was still there.
- The handler code was never touched. The seam is at the **edge** of the handler (the `Depends(...)`), not inside it.

In a real test suite you'd put this in a pytest fixture (chapter 7). The idea is the same: build a clean `InMemoryAssetRepo`, install it as an override, run the test, clear. Each test gets a fresh repo with deterministic data — no `BEGIN TRANSACTION; ROLLBACK;` rituals, no flakes from a shared database.

## Key Takeaways

- **A repository is a thin abstraction between the handler and the data source.** The handler talks to the Protocol; concrete repos implement it.
- **`typing.Protocol` is duck-typing with a name.** No inheritance, no `ABC`, no runtime cost. Type checkers verify shape.
- **Inject via `Depends`.** The handler declares it wants an `AssetRepo`; the provider decides which one (singleton, per-request, per-tenant, feature-flagged).
- **`app.dependency_overrides`** swaps the dep for a test or environment. No globals, no monkeypatching, no `if testing:` branches.
- **The shape of the repo IS the contract.** If two repos can't be swapped 1:1, the Protocol is the wrong shape — not the handler's problem.
- **Capstone tie-in**: the portfolio API will define `AssetRepo`, `PortfolioRepo`, `HoldingRepo` protocols, with SQLite as the dev backend and a Postgres adapter as the prod target. Tests use in-memory implementations.

## Exercises

**1. Postgres skeleton.** Add a class `PostgresAssetRepo` that has the same four methods with the right signatures. You don't need to actually connect — just `raise NotImplementedError` in each body. Confirm it type-checks against `AssetRepo` (a static checker would pass; at runtime you can do `repo: AssetRepo = PostgresAssetRepo()` without error).

**2. Override at the test level.** Write a small `make_client()` helper that returns a fresh `TestClient` whose `get_repo` is overridden to a brand-new `InMemoryAssetRepo`. Each call returns a clean client. Use it to write two tests that don't see each other's data — proving test isolation comes for free with this pattern.

**3. Repo with a `yield` dep.** Convert `get_repo` to a `yield` dep that opens a `SQLiteAssetRepo` per request, yields it, and closes the connection on teardown. Verify with a print statement in `__init__` and a `close()` method that the connection lifecycle tracks the request lifecycle 1:1.